# Sorting, Ranking and Apply

Three skills that turn a plain table into **insight and features**:

- **Sorting**: put rows in a meaningful order (leaderboards, top-N, time order)
- **Ranking**: give every row a position or percentile (and do it per group)
- **Apply / map / transform**: run your own logic on columns and rows, and know when **not** to
- **Sequential operations**: `cumsum`, `diff`, `shift`, `rolling` (the base of time-series features for ML)

**Contents**
1. Our datasets
2. Sorting
3. Ranking
4. `map`, `apply`, `DataFrame.map`, `agg`, `transform`
5. Speed: vectorized vs `apply` vs loops (measured)
6. Cumulative, `diff`, `shift`, `rolling`
7. Mini AI/ML example: features for forecasting
8. Pandas 3.x notes
9. Common mistakes
10. Interview questions
11. Practice questions with solutions
12. Cheat sheet

In [66]:
import time
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. Our datasets

Notice the **ties** on purpose: Neha and Pooja both have 88, Karan and Rohan both have 58. Ties are where sorting and ranking get interesting.

In [67]:
df = pd.DataFrame({
    "name":       ["Ravi", "Asha", "Karan", "Neha", "Amit", "Sara", "Dev", "Pooja", "Isha", "Rohan"],
    "city":       ["Indore", "Bhopal", "Delhi", "Indore", "Pune", "Bhopal", "Delhi", "Indore", "Pune", "Delhi"],
    "hours":      [5.5, 7.0, 3.0, 6.5, 4.0, 8.0, 2.5, 6.0, 5.0, 3.5],
    "score":      [85, 92, 58, 88, 64, 97, 45, 88, 70, 58],
    "attendance": [80, 90, 65, 85, 70, 95, 60, 88, np.nan, 72],
})
df

,name,city,hours,score,attendance
0,Ravi,Indore,5.5,85,80.0
1,Asha,Bhopal,7.0,92,90.0
2,Karan,Delhi,3.0,58,65.0
3,Neha,Indore,6.5,88,85.0
4,Amit,Pune,4.0,64,70.0
5,Sara,Bhopal,8.0,97,95.0
6,Dev,Delhi,2.5,45,60.0
7,Pooja,Indore,6.0,88,88.0
8,Isha,Pune,5.0,70,NaN
9,Rohan,Delhi,3.5,58,72.0


---
# 2. Sorting

## 2.1 `sort_values()`

In [68]:
df.sort_values("score")                          # ascending (default)

,name,city,hours,score,attendance
6,Dev,Delhi,2.5,45,60.0
2,Karan,Delhi,3.0,58,65.0
9,Rohan,Delhi,3.5,58,72.0
4,Amit,Pune,4.0,64,70.0
8,Isha,Pune,5.0,70,NaN
0,Ravi,Indore,5.5,85,80.0
3,Neha,Indore,6.5,88,85.0
7,Pooja,Indore,6.0,88,88.0
1,Asha,Bhopal,7.0,92,90.0
5,Sara,Bhopal,8.0,97,95.0


In [69]:
df.sort_values("score", ascending=False)         # highest first

,name,city,hours,score,attendance
5,Sara,Bhopal,8.0,97,95.0
1,Asha,Bhopal,7.0,92,90.0
3,Neha,Indore,6.5,88,85.0
7,Pooja,Indore,6.0,88,88.0
0,Ravi,Indore,5.5,85,80.0
8,Isha,Pune,5.0,70,NaN
4,Amit,Pune,4.0,64,70.0
2,Karan,Delhi,3.0,58,65.0
9,Rohan,Delhi,3.5,58,72.0
6,Dev,Delhi,2.5,45,60.0


In [70]:
# Several columns: sort by city A-Z, and inside each city by score high -> low
df.sort_values(["city", "score"], ascending=[True, False])

,name,city,hours,score,attendance
5,Sara,Bhopal,8.0,97,95.0
1,Asha,Bhopal,7.0,92,90.0
2,Karan,Delhi,3.0,58,65.0
9,Rohan,Delhi,3.5,58,72.0
6,Dev,Delhi,2.5,45,60.0
3,Neha,Indore,6.5,88,85.0
7,Pooja,Indore,6.0,88,88.0
0,Ravi,Indore,5.5,85,80.0
8,Isha,Pune,5.0,70,NaN
4,Amit,Pune,4.0,64,70.0


In [71]:
# The old index travels with the rows. ignore_index=True gives a fresh 0..n-1 index
df.sort_values("score", ascending=False, ignore_index=True).head(4)

,name,city,hours,score,attendance
0,Sara,Bhopal,8.0,97,95.0
1,Asha,Bhopal,7.0,92,90.0
2,Neha,Indore,6.5,88,85.0
3,Pooja,Indore,6.0,88,88.0


> Remember chapter 05: after sorting, `loc[0]` means "the row with label 0", not "the first row". Use `iloc[0]` or `ignore_index=True`.

## 2.2 Missing values in sorting

By default NaN goes **last**, for both ascending and descending.

In [72]:
print(df.sort_values("attendance")[["name", "attendance"]].tail(3))
print()
print(df.sort_values("attendance", na_position="first")[["name", "attendance"]].head(3))

   name  attendance
1  Asha        90.0
5  Sara        95.0
8  Isha         NaN

    name  attendance
8   Isha         NaN
6    Dev        60.0
2  Karan        65.0


## 2.3 Ties and stability

When two rows have the same value, what order do they get? A **stable** sort keeps their **original order**. This matters for reproducible results.

- Multi-column sorts are stable by design.
- A single-column `sort_values` uses `quicksort` by default, which is **not guaranteed stable**. Use `kind="stable"` when ties matter.

In [73]:
df.sort_values("score", ascending=False, kind="stable")[["name", "score"]]
# Neha comes before Pooja (both 88) because Neha was earlier in the original table

,name,score
5,Sara,97
1,Asha,92
3,Neha,88
7,Pooja,88
0,Ravi,85
8,Isha,70
4,Amit,64
2,Karan,58
9,Rohan,58
6,Dev,45


## 2.4 Custom sort order

### Case-insensitive text sort (`key=`)

Normal text sorting puts capital letters before small letters (`"Zoya"` before `"amit"`).

In [74]:
names = pd.Series(["bob", "Alice", "carol", "Dave"])
print(names.sort_values().tolist())                               # capitals first
print(names.sort_values(key=lambda s: s.str.lower()).tolist())    # real alphabetical order

['Alice', 'Dave', 'bob', 'carol']
['Alice', 'bob', 'carol', 'Dave']


### Your own order (Low < Medium < High)

Alphabetical order would give High, Low, Medium. Two ways to get the logical order:

In [75]:
levels = pd.DataFrame({"task": ["a", "b", "c", "d"], "priority": ["Medium", "High", "Low", "High"]})

# Way 1: key= with a dictionary
order = {"Low": 0, "Medium": 1, "High": 2}
print(levels.sort_values("priority", key=lambda s: s.map(order))["priority"].tolist())

# Way 2: ordered Categorical dtype (better when the order is used many times)
levels["priority"] = pd.Categorical(levels["priority"], categories=["Low", "Medium", "High"], ordered=True)
print(levels.sort_values("priority")["priority"].tolist())
print(levels["priority"].max())               # ordered categories also support max/min

['Low', 'Medium', 'High', 'High']
['Low', 'Medium', 'High', 'High']
High


## 2.5 `sort_index()`

In [76]:
by_name = df.set_index("name")
by_name.sort_index().head(4)

,city,hours,score,attendance
name,,,,
Amit,Pune,4.0,64,70.0
Asha,Bhopal,7.0,92,90.0
Dev,Delhi,2.5,45,60.0
Isha,Pune,5.0,70,NaN


In [77]:
by_name.sort_index(ascending=False).head(3)

,city,hours,score,attendance
name,,,,
Sara,Bhopal,8.0,97,95.0
Rohan,Delhi,3.5,58,72.0
Ravi,Indore,5.5,85,80.0


## 2.6 `nlargest` and `nsmallest`

Faster and shorter than a full sort when you only need the top N.

In [78]:
df.nlargest(3, "score")[["name", "score"]]

,name,score
5,Sara,97
1,Asha,92
3,Neha,88


In [79]:
# Ties at the border: keep="first" (default) cuts the tie, keep="all" keeps every tied row
print(df.nsmallest(2, "score", keep="first")[["name", "score"]])     # 2 rows: the tie is cut
print()
print(df.nsmallest(2, "score", keep="all")[["name", "score"]])       # 3 rows: both tied rows stay

    name  score
6    Dev     45
2  Karan     58

    name  score
6    Dev     45
2  Karan     58
9  Rohan     58


In [80]:
# Several columns: highest score, ties broken by more hours
df.nlargest(4, ["score", "hours"])[["name", "score", "hours"]]

,name,score,hours
5,Sara,97,8.0
1,Asha,92,7.0
3,Neha,88,6.5
7,Pooja,88,6.0


In [81]:
# Series version, and the position/label of the best value
print(df["score"].nlargest(2).tolist())
print(df["score"].idxmax(), df.loc[df["score"].idxmax(), "name"])

[97, 92]
5 Sara


## 2.7 Top N per group (sort first, then take from each group)

In [82]:
# Best 2 students in every city
(
    df.sort_values("score", ascending=False)
      .groupby("city")
      .head(2)
      .sort_values(["city", "score"], ascending=[True, False])
      [["city", "name", "score"]]
)

,city,name,score
5,Bhopal,Sara,97
1,Bhopal,Asha,92
2,Delhi,Karan,58
9,Delhi,Rohan,58
3,Indore,Neha,88
7,Indore,Pooja,88
8,Pune,Isha,70
4,Pune,Amit,64


---
# 3. Ranking

Sorting **reorders** the rows. Ranking **adds a number** (the position) to each row and leaves the order alone. That number is a ready-made feature.

## 3.1 `rank()` and the 5 tie methods

Say two students share the top score. Are they both 1st? Is the next one 2nd or 3rd? That is the `method`.

In [83]:
methods = ["average", "min", "max", "first", "dense"]

rank_table = pd.DataFrame({m: df["score"].rank(method=m, ascending=False) for m in methods})
rank_table.insert(0, "score", df["score"])
rank_table.insert(0, "name", df["name"])
rank_table.sort_values("score", ascending=False)

,name,score,average,min,max,first,dense
5,Sara,97,1.0,1.0,1.0,1.0,1.0
1,Asha,92,2.0,2.0,2.0,2.0,2.0
3,Neha,88,3.5,3.0,4.0,3.0,3.0
7,Pooja,88,3.5,3.0,4.0,4.0,3.0
0,Ravi,85,5.0,5.0,5.0,5.0,4.0
8,Isha,70,6.0,6.0,6.0,6.0,5.0
4,Amit,64,7.0,7.0,7.0,7.0,6.0
2,Karan,58,8.5,8.0,9.0,8.0,7.0
9,Rohan,58,8.5,8.0,9.0,9.0,7.0
6,Dev,45,10.0,10.0,10.0,10.0,8.0


| `method` | Ties get | Example for scores 97, 92, 88, 88, 85 | Used for |
|---|---|---|---|
| `average` (default) | the average rank | 1, 2, 3.5, 3.5, 5 | statistics (Spearman correlation) |
| `min` | the lowest (best) rank | 1, 2, 3, 3, 5 | **competition ranking** ("1224") |
| `max` | the highest (worst) rank | 1, 2, 4, 4, 5 | rarely |
| `first` | by order of appearance (no ties at all) | 1, 2, 3, 4, 5 | unique rank, top-N selection |
| `dense` | the same rank, **no gaps** | 1, 2, 3, 3, 4 | levels / tiers ("1223") |

`ascending=False` makes the **highest** value rank 1. The default `ascending=True` gives the lowest value rank 1.

In [84]:
# Leaderboard with competition ranking
board = df.assign(rank=df["score"].rank(method="min", ascending=False).astype(int))
board.sort_values("rank")[["rank", "name", "score"]]

,rank,name,score
5,1,Sara,97
1,2,Asha,92
3,3,Neha,88
7,3,Pooja,88
0,5,Ravi,85
8,6,Isha,70
4,7,Amit,64
2,8,Karan,58
9,8,Rohan,58
6,10,Dev,45


## 3.2 Percentile rank: `pct=True`

Gives a value between 0 and 1 representing the row's **relative rank within the dataset**. Higher values get higher percentile positions.

With ties, pandas uses the selected ranking method (default: `average`), so it is not always exactly the fraction of values at or below the row.

This is useful as a **scale-free feature** because it puts values on a common 0-1 scale.

In [85]:
df["score_pct"] = df["score"].rank(pct=True).round(2)
df[["name", "score", "score_pct"]].sort_values("score")

,name,score,score_pct
6,Dev,45,0.10
2,Karan,58,0.25
9,Rohan,58,0.25
4,Amit,64,0.40
8,Isha,70,0.50
0,Ravi,85,0.60
3,Neha,88,0.75
7,Pooja,88,0.75
1,Asha,92,0.90
5,Sara,97,1.00


## 3.3 Ranking with missing values

In [86]:
att = df["attendance"]
print(att.rank(ascending=False).tolist())                         # NaN stays NaN (default, "keep")
print(att.rank(ascending=False, na_option="bottom").tolist())     # NaN ranked last

[5.0, 2.0, 8.0, 4.0, 7.0, 1.0, 9.0, 3.0, nan, 6.0]
[5.0, 2.0, 8.0, 4.0, 7.0, 1.0, 9.0, 3.0, 10.0, 6.0]


## 3.4 Rank inside groups

`groupby(...).rank()` restarts the ranking for **every group**. Perfect for "best in each city".

In [87]:
df["rank_in_city"] = (
    df.groupby("city")["score"]
      .rank(method="min", ascending=False)
      .astype("Int64")
)
df.sort_values(["city", "rank_in_city"])[["city", "name", "score", "rank_in_city"]]

,city,name,score,rank_in_city
5,Bhopal,Sara,97,1
1,Bhopal,Asha,92,2
2,Delhi,Karan,58,1
9,Delhi,Rohan,58,1
6,Delhi,Dev,45,3
3,Indore,Neha,88,1
7,Indore,Pooja,88,1
0,Indore,Ravi,85,3
8,Pune,Isha,70,1
4,Pune,Amit,64,2


In [88]:
# Top 1 per city using rank (method="first" guarantees exactly one winner even with ties)
first_rank = df.groupby("city")["score"].rank(method="first", ascending=False)
df[first_rank == 1][["city", "name", "score"]]

,city,name,score
2,Delhi,Karan,58
3,Indore,Neha,88
5,Bhopal,Sara,97
8,Pune,Isha,70


## `sort_values` vs `rank` vs `nlargest`

| | Output | Use when |
|---|---|---|
| `sort_values` | the table in a new **order** | you want to look at / export a sorted table |
| `rank` | a **number column**, order unchanged | you need position as a feature, or per-group position |
| `nlargest` | only the **top N rows** | you just need the best few |

---
# 4. `map`, `apply`, `DataFrame.map`, `agg`, `transform`

Pandas has built-in methods for most things. When you need **your own logic**, these are the tools.

## 4.1 `Series.map()`: element by element

In [89]:
# with a dictionary (most common: translate values)
print(df["city"].map({"Indore": "MP", "Bhopal": "MP", "Delhi": "DL", "Pune": "MH"}).tolist())

# with a function
print(df["score"].map(lambda x: "pass" if x >= 60 else "fail").tolist())

['MP', 'MP', 'DL', 'MP', 'MH', 'MP', 'DL', 'MP', 'MH', 'DL']
['pass', 'pass', 'fail', 'pass', 'pass', 'pass', 'fail', 'pass', 'pass', 'fail']


In [90]:
# Values missing in the dictionary become NaN; na_action="ignore" skips NaN inputs
codes = pd.Series(["A", "B", None, "Z"])
print(codes.map({"A": 1, "B": 2}).tolist())
print(codes.map(lambda x: x.lower(), na_action="ignore").tolist())

[1.0, 2.0, nan, nan]
['a', 'b', nan, 'z']


## 4.2 `Series.apply()`: a function on every value

For simple one-value-in, one-value-out jobs `map` and `apply` give the same result. `apply` can also pass extra arguments:

In [91]:
def grade(score, pass_mark=60, top_mark=90):
    if score >= top_mark:
        return "A"
    if score >= pass_mark:
        return "B"
    return "F"

print(df["score"].apply(grade).tolist())
print(df["score"].apply(grade, pass_mark=50, top_mark=85).tolist())      # extra keyword arguments

['B', 'A', 'F', 'B', 'B', 'A', 'F', 'B', 'B', 'F']
['A', 'A', 'B', 'A', 'B', 'A', 'F', 'A', 'B', 'B']


## 4.3 `DataFrame.apply()`: along columns or rows

- `axis=0` (default): the function receives **one column at a time**
- `axis=1`: the function receives **one row at a time**

In [92]:
nums = df[["hours", "score", "attendance"]]

print(nums.apply(lambda col: col.max() - col.min()))          # one number per column (range)

hours          5.5
score         52.0
attendance    35.0
dtype: float64


In [93]:
# axis=1: row by row (slow on big tables, use for logic that really needs several columns)
df["summary"] = df.apply(lambda r: f"{r['name']} ({r['city']}): {r['score']}", axis=1)
df["summary"].head(3).tolist()

['Ravi (Indore): 85', 'Asha (Bhopal): 92', 'Karan (Delhi): 58']

In [94]:
# A row function can return several values -> several new columns
def effort(row):
    return pd.Series({
        "study_minutes": row["hours"] * 60,
        "efficiency": round(row["score"] / row["hours"], 1),
    })

df[["name"]].join(df.apply(effort, axis=1)).head(4)

,name,study_minutes,efficiency
0,Ravi,330.0,15.5
1,Asha,420.0,13.1
2,Karan,180.0,19.3
3,Neha,390.0,13.5


## 4.4 `DataFrame.map()`: every single cell

Applies a function to **each element** of the whole DataFrame. (In older tutorials this is called `applymap`, which **no longer exists in pandas 3**.)

In [95]:
nums.head(3).map(lambda x: f"{x:.1f}")          # format every number as text

,hours,score,attendance
0,5.5,85.0,80.0
1,7.0,92.0,90.0
2,3.0,58.0,65.0


In [96]:
print(hasattr(pd.DataFrame, "applymap"), hasattr(pd.DataFrame, "map"))

False True


## 4.5 `agg` vs `transform` vs `apply` (the shape rule)

| Method | Returns | Shape |
|---|---|---|
| `agg` | one value per group/column | **shrinks** (one row per group) |
| `transform` | one value per original row | **same shape** as the input |
| `apply` | anything | depends on your function |

`transform` is the one for **features**: it keeps all rows and adds group knowledge to each of them.

In [97]:
g = df.groupby("city")["score"]

print(g.agg("mean").round(1))                       # 4 rows (one per city)
print()
print(g.transform("mean").round(1).tolist())        # 10 values (one per student)

city
Bhopal    94.5
Delhi     53.7
Indore    87.0
Pune      67.0
Name: score, dtype: float64

[87.0, 94.5, 53.7, 87.0, 67.0, 94.5, 53.7, 87.0, 67.0, 53.7]


In [98]:
# Typical features built with transform
df["city_avg"] = g.transform("mean").round(1)
df["vs_city_avg"] = (df["score"] - df["city_avg"]).round(1)

# z-score inside each city: (value - group mean) / group std
df["score_z_city"] = g.transform(lambda s: (s - s.mean()) / s.std()).round(2)

df[["name", "city", "score", "city_avg", "vs_city_avg", "score_z_city"]]

,name,city,score,city_avg,vs_city_avg,score_z_city
0,Ravi,Indore,85,87.0,-2.0,-1.15
1,Asha,Bhopal,92,94.5,-2.5,-0.71
2,Karan,Delhi,58,53.7,4.3,0.58
3,Neha,Indore,88,87.0,1.0,0.58
4,Amit,Pune,64,67.0,-3.0,-0.71
5,Sara,Bhopal,97,94.5,2.5,0.71
6,Dev,Delhi,45,53.7,-8.7,-1.15
7,Pooja,Indore,88,87.0,1.0,0.58
8,Isha,Pune,70,67.0,3.0,0.71
9,Rohan,Delhi,58,53.7,4.3,0.58


A group with **one** row has no standard deviation, so its z-score is NaN. Handle that (fill with 0) before modelling.

## 4.6 `pipe()`: your own function inside a chain

In [99]:
def add_grade(d, pass_mark=60):
    d = d.copy()
    d["result"] = np.where(d["score"] >= pass_mark, "Pass", "Fail")
    return d

df.pipe(add_grade, pass_mark=70)[["name", "score", "result"]].head(4)

,name,score,result
0,Ravi,85,Pass
1,Asha,92,Pass
2,Karan,58,Fail
3,Neha,88,Pass


---
# 5. Speed: vectorized vs `apply` vs loops (measured)

Same calculation, five ways, on 20,000 rows: `price * qty * (1 - discount%)`.

In [100]:
rng = np.random.default_rng(1)
n = 20_000
big = pd.DataFrame({
    "price": rng.random(n) * 100,
    "qty": rng.integers(1, 10, n),
    "discount": rng.integers(0, 30, n),
})

def total(price, qty, discount):
    return price * qty * (1 - discount / 100)

times, results = {}, {}

start = time.perf_counter()
results["iterrows loop"] = [total(r["price"], r["qty"], r["discount"]) for _, r in big.iterrows()]
times["iterrows loop"] = time.perf_counter() - start

start = time.perf_counter()
results["apply(axis=1)"] = big.apply(lambda r: total(r["price"], r["qty"], r["discount"]), axis=1)
times["apply(axis=1)"] = time.perf_counter() - start

start = time.perf_counter()
results["itertuples loop"] = [total(r.price, r.qty, r.discount) for r in big.itertuples()]
times["itertuples loop"] = time.perf_counter() - start

start = time.perf_counter()
results["zip loop"] = [total(p, q, d) for p, q, d in zip(big["price"], big["qty"], big["discount"])]
times["zip loop"] = time.perf_counter() - start

start = time.perf_counter()
results["vectorized"] = big["price"] * big["qty"] * (1 - big["discount"] / 100)
times["vectorized"] = time.perf_counter() - start

speed = pd.DataFrame({"seconds": times})
speed["times_slower_than_vectorized"] = (speed["seconds"] / speed.loc["vectorized", "seconds"]).round(0)
speed.sort_values("seconds")

,seconds,times_slower_than_vectorized
vectorized,0.000290,1.0
zip loop,0.004548,16.0
itertuples loop,0.008414,29.0
apply(axis=1),0.073410,253.0
iterrows loop,0.198948,685.0


In [101]:
# all five methods give the same numbers
print(all(np.allclose(np.asarray(v), results["vectorized"].to_numpy()) for v in results.values()))

True


**The typical order of speed (fastest to slowest). The first, third, fourth and fifth were measured above:**

1. **Vectorized column operations** (`a * b`, `np.where`, `np.select`, built-in methods)
2. `map` with a dict, then `apply` on one column
3. `zip` / `itertuples` loops
4. `apply(axis=1)`
5. `iterrows` (slowest, builds a Series for every row)

Rules:
- Can you write it with column operations, `np.where` or `np.select`? Do that.
- Need row logic? Try `zip(df["a"], df["b"])`, it is simple and fast.
- Use `apply(axis=1)` only for small tables or logic that truly cannot be vectorized.
- Never use `iterrows` to compute values.

---
# 6. Cumulative, `diff`, `shift`, `rolling`

These operations depend on **order**. That is why **sorting comes first**.

## 6.1 Our time-ordered data: daily units sold in two stores

In [102]:
rng = np.random.default_rng(5)
days = pd.date_range("2026-03-01", periods=8)

sales = pd.DataFrame({
    "date":  list(days) * 2,
    "store": ["Indore"] * 8 + ["Pune"] * 8,
    "units": rng.integers(20, 60, 16),
})

# real data often arrives in random order: shuffle to copy that
sales = sales.sample(frac=1, random_state=3).reset_index(drop=True)
sales.head(6)

,date,store,units
0,2026-03-08,Indore,31
1,2026-03-06,Pune,36
2,2026-03-05,Indore,38
3,2026-03-02,Indore,52
4,2026-03-07,Indore,45
5,2026-03-03,Indore,20


## 6.2 Always sort first

In [103]:
# WRONG: shift on shuffled data compares a day with a random other day
wrong = sales.assign(yesterday=sales["units"].shift(1))
print(wrong.head(4))

        date   store  units  yesterday
0 2026-03-08  Indore     31        NaN
1 2026-03-06    Pune     36       31.0
2 2026-03-05  Indore     38       36.0
3 2026-03-02  Indore     52       38.0


In [104]:
# RIGHT: sort by store and date, then work
sales = sales.sort_values(["store", "date"]).reset_index(drop=True)
sales.head(10)

,date,store,units
0,2026-03-01,Indore,46
1,2026-03-02,Indore,52
2,2026-03-03,Indore,20
3,2026-03-04,Indore,52
4,2026-03-05,Indore,38
5,2026-03-06,Indore,40
6,2026-03-07,Indore,45
7,2026-03-08,Indore,31
8,2026-03-01,Pune,59
9,2026-03-02,Pune,22


## 6.3 `cumsum`, `cummax`, `cummin`, `cumprod`

In [105]:
s = pd.Series([3, 1, 4, 1, 5])
print("cumsum :", s.cumsum().tolist())
print("cummax :", s.cummax().tolist())
print("cummin :", s.cummin().tolist())
print("cumprod:", s.cumprod().tolist())

cumsum : [3, 4, 8, 9, 14]
cummax : [3, 3, 4, 4, 5]
cummin : [3, 1, 1, 1, 1]
cumprod: [3, 3, 12, 12, 60]


In [106]:
# Running total INSIDE each store (the groupby restarts the total for every store)
sales["running_total"] = sales.groupby("store")["units"].cumsum()
sales["best_so_far"] = sales.groupby("store")["units"].cummax()
sales[sales["store"] == "Indore"]

,date,store,units,running_total,best_so_far
0,2026-03-01,Indore,46,46,46
1,2026-03-02,Indore,52,98,52
2,2026-03-03,Indore,20,118,52
3,2026-03-04,Indore,52,170,52
4,2026-03-05,Indore,38,208,52
5,2026-03-06,Indore,40,248,52
6,2026-03-07,Indore,45,293,52
7,2026-03-08,Indore,31,324,52


## 6.4 `diff` and `pct_change`: change from the previous row

In [107]:
sales["change"] = sales.groupby("store")["units"].diff()
sales["change_pct"] = (sales.groupby("store")["units"].pct_change() * 100).round(1)
sales[sales["store"] == "Pune"][["date", "units", "change", "change_pct"]]

,date,units,change,change_pct
8,2026-03-01,59,NaN,NaN
9,2026-03-02,22,-37.0,-62.7
10,2026-03-03,31,9.0,40.9
11,2026-03-04,35,4.0,12.9
12,2026-03-05,42,7.0,20.0
13,2026-03-06,36,-6.0,-14.3
14,2026-03-07,25,-11.0,-30.6
15,2026-03-08,21,-4.0,-16.0


The **first row of every store is NaN**, because there is no previous day. The groupby makes sure Pune's first day is not compared with Indore's last day.

## 6.5 `shift`: lag and lead features

- `shift(1)` = the value **one row before** (yesterday)
- `shift(-1)` = the value **one row after** (tomorrow)

In [108]:
g = sales.groupby("store")["units"]
sales["lag_1"] = g.shift(1)         # yesterday's units
sales["lag_2"] = g.shift(2)         # day before yesterday
sales["lead_1"] = g.shift(-1)       # tomorrow's units (a "label", never use as a feature!)
sales["beat_yesterday"] = sales["units"] > sales["lag_1"]

sales[sales["store"] == "Indore"][["date", "units", "lag_1", "lag_2", "lead_1", "beat_yesterday"]]

,date,units,lag_1,lag_2,lead_1,beat_yesterday
0,2026-03-01,46,NaN,NaN,52.0,False
1,2026-03-02,52,46.0,NaN,20.0,True
2,2026-03-03,20,52.0,46.0,52.0,False
3,2026-03-04,52,20.0,52.0,38.0,True
4,2026-03-05,38,52.0,20.0,40.0,False
5,2026-03-06,40,38.0,52.0,45.0,True
6,2026-03-07,45,40.0,38.0,31.0,True
7,2026-03-08,31,45.0,40.0,NaN,False


## 6.6 `rolling`: a moving window

`rolling(3).mean()` = the average of the **current and the 2 previous rows**. It smooths noise.

In [109]:
s = pd.Series([10, 20, 30, 40, 50, 60])
print("rolling(3).mean()                :", s.rolling(3).mean().tolist())              # first 2 are NaN
print("rolling(3, min_periods=1).mean() :", s.rolling(3, min_periods=1).mean().tolist())   # use what is available
print("rolling(3).sum()                 :", s.rolling(3).sum().tolist())
print("rolling(3).max()                 :", s.rolling(3).max().tolist())

rolling(3).mean()                : [nan, nan, 20.0, 30.0, 40.0, 50.0]
rolling(3, min_periods=1).mean() : [10.0, 15.0, 20.0, 30.0, 40.0, 50.0]
rolling(3).sum()                 : [nan, nan, 60.0, 90.0, 120.0, 150.0]
rolling(3).max()                 : [nan, nan, 30.0, 40.0, 50.0, 60.0]


In [110]:
# Rolling inside each group: use transform with a lambda
sales["roll3"] = sales.groupby("store")["units"].transform(lambda x: x.rolling(3, min_periods=1).mean()).round(1)
sales[sales["store"] == "Indore"][["date", "units", "roll3"]]

,date,units,roll3
0,2026-03-01,46,46.0
1,2026-03-02,52,49.0
2,2026-03-03,20,39.3
3,2026-03-04,52,41.3
4,2026-03-05,38,36.7
5,2026-03-06,40,43.3
6,2026-03-07,45,41.0
7,2026-03-08,31,38.7


In [111]:
# Time-based window: "last 3 days" (needs a date index; handles missing days correctly)
indore = sales[sales["store"] == "Indore"].set_index("date")["units"]
indore.rolling("3D").mean().round(1).head(5)

date
2026-03-01    46.0
2026-03-02    49.0
2026-03-03    39.3
2026-03-04    41.3
2026-03-05    36.7
Name: units, dtype: float64

## 6.7 `expanding` and `ewm`

- `expanding().mean()`: average of **everything so far** (window grows)
- `ewm(span=3).mean()`: exponentially weighted, **recent values matter more**

In [112]:
s = pd.Series([10, 20, 30, 40, 50, 60])
print("expanding mean:", s.expanding().mean().tolist())
print("ewm mean      :", s.ewm(span=3).mean().round(1).tolist())

expanding mean: [10.0, 15.0, 20.0, 25.0, 30.0, 35.0]
ewm mean      : [10.0, 16.7, 24.3, 32.7, 41.6, 51.0]


---
# 7. Mini AI/ML example: features for forecasting

Goal: predict **today's units** for a store. We can only use information that was **known before today**.

**The leakage rule for time features:**
- `rolling(3).mean()` on `units` includes **today's own value**. Today's value is the **answer**, so the model would just "see" it. That is data leakage.
- Fix: `shift(1)` first, then roll. Now the window contains only **past** days.

In [113]:
ml = sales[["date", "store", "units"]].copy()
g = ml.groupby("store")["units"]

# Past-only features
ml["lag_1"] = g.shift(1)
ml["lag_2"] = g.shift(2)
ml["roll3_past"] = g.transform(lambda x: x.shift(1).rolling(3).mean()).round(2)       # shift THEN roll
ml["change_prev"] = g.shift(1) - g.shift(2)                                            # yesterday's change

# WRONG version for comparison (includes today's value)
ml["roll3_LEAKY"] = g.transform(lambda x: x.rolling(3).mean()).round(2)

# Calendar and cross-store features
ml["day_of_week"] = ml["date"].dt.dayofweek
ml["store_rank_today_LEAKY"] = ml.groupby("date")["units"].rank(ascending=False).astype(int)   # uses TODAY's units

ml[ml["store"] == "Indore"].drop(columns="store")

,date,units,lag_1,lag_2,roll3_past,change_prev,roll3_LEAKY,day_of_week,store_rank_today_LEAKY
0,2026-03-01,46,NaN,NaN,NaN,NaN,NaN,6,2
1,2026-03-02,52,46.0,NaN,NaN,NaN,NaN,0,1
2,2026-03-03,20,52.0,46.0,NaN,6.0,39.33,1,2
3,2026-03-04,52,20.0,52.0,39.33,-32.0,41.33,2,1
4,2026-03-05,38,52.0,20.0,41.33,32.0,36.67,3,2
5,2026-03-06,40,38.0,52.0,36.67,-14.0,43.33,4,1
6,2026-03-07,45,40.0,38.0,43.33,2.0,41.00,5,1
7,2026-03-08,31,45.0,40.0,41.00,5.0,38.67,6,1


**Check yourself:**
- `roll3_past` on a given day = average of the previous 3 days. Good.
- `roll3_LEAKY` includes the current day. Do **not** use it.
- `store_rank_today_LEAKY` also uses **today's** `units` (compared with the other store), so it is leaky too. It is fine for a report, **not** for a feature. Features must use only past data.
- The first rows of each store have NaN lags (no history). Drop them or fill them (chapter 07).

In [114]:
# Time-based split: train on the EARLIER days, test on the LATER days (never a random split for time series)
features = ["lag_1", "lag_2", "roll3_past", "change_prev", "day_of_week"]
model_data = ml.dropna(subset=features)                      # drop days that have no history yet

dates = np.sort(model_data["date"].unique())
test_start = dates[-2]                                        # the last 2 days become the test set

train = model_data[model_data["date"] < test_start]
test = model_data[model_data["date"] >= test_start]

print("train:", train["date"].min().date(), "->", train["date"].max().date(), "| rows:", len(train))
print("test :", test["date"].min().date(), "->", test["date"].max().date(), "| rows:", len(test))

X_train, y_train = train[features], train["units"]
X_test, y_test = test[features], test["units"]

train: 2026-03-04 -> 2026-03-06 | rows: 6
test : 2026-03-07 -> 2026-03-08 | rows: 4


### The same ideas on the student table: rank-based features

In [115]:
students = df[["name", "city", "hours", "score"]].copy()

students["score_percentile"] = students["score"].rank(pct=True).round(2)                     # overall percentile
students["hours_rank_in_city"] = students.groupby("city")["hours"].rank(ascending=False)      # effort rank in city
students["hours_vs_city_avg"] = (students["hours"] - students.groupby("city")["hours"].transform("mean")).round(2)

students.head(6)

,name,city,hours,score,score_percentile,hours_rank_in_city,hours_vs_city_avg
0,Ravi,Indore,5.5,85,0.60,3.0,-0.5
1,Asha,Bhopal,7.0,92,0.90,2.0,-0.5
2,Karan,Delhi,3.0,58,0.25,2.0,0.0
3,Neha,Indore,6.5,88,0.75,1.0,0.5
4,Amit,Pune,4.0,64,0.40,2.0,-0.5
5,Sara,Bhopal,8.0,97,1.00,1.0,0.5


---
# 8. Pandas 3.x notes

- **`applymap` is gone.** Use `DataFrame.map` for element-wise functions.
- `groupby(...).apply(func)` passes the group **without** the grouping column(s). If you need the group name, use `g.name` or `transform`/`agg` instead.
- `pct_change()` no longer fills missing values silently (the old `fill_method` behaviour was removed). The first value is NaN.
- `sort_values` on a single column is not stable by default: add `kind="stable"` when ties matter.
- Text columns use the `str` dtype: `sort_values`, `rank` (on text), `map` and `apply` work as before.
- Sorting and ranking return **new** objects. `df.sort_values(...)` alone does not change `df`.

---
# 9. Common mistakes

| Mistake | Fix |
|---|---|
| `df.sort_values("a")` and expecting `df` to change | `df = df.sort_values("a")` |
| Using `loc[0]` after sorting for "first row" | `iloc[0]` or `ignore_index=True` |
| Ties give random order between runs | `kind="stable"` or sort on a second column |
| Using `rank()` default and expecting 1, 2, 3 with ties | Choose `method` (`min`, `dense`, `first`) |
| `rank()` ascending when "high is good" | `ascending=False` |
| Applying `shift`/`rolling` on unsorted data | Sort by group and time first |
| `shift`/`rolling` across groups | `groupby(...)` first, so each group restarts |
| `rolling(3).mean()` as a feature for today's value | `shift(1)` first (leakage) |
| Random train/test split on time-ordered data | Split by time |
| `apply(axis=1)` for simple arithmetic | Vectorized columns, `np.where`, `np.select` |
| `iterrows()` to compute values | Vectorize, or `zip`/`itertuples` if you must loop |
| `applymap` | `DataFrame.map` |
| Groupby `transform` with a function that returns a different length | `transform` must return the same length as the group |
| Forgetting NaN z-scores in groups of one row | Fill them (for example with 0) |

---
# 10. Interview questions

**Q1. `sort_values` vs `sort_index`?**
`sort_values` sorts by the content of columns, `sort_index` sorts by the index labels.

**Q2. How do you sort by multiple columns with different directions?**
`df.sort_values(["a", "b"], ascending=[True, False])`.

**Q3. How do you sort in a custom order (Low, Medium, High)?**
Use an ordered `Categorical` dtype, or `sort_values(key=lambda s: s.map(order_dict))`.

**Q4. Explain the `method` options of `rank`.**
`average` gives ties the mean rank, `min` gives the lowest rank (competition ranking 1-2-2-4), `max` the highest, `first` breaks ties by order of appearance, `dense` gives ties the same rank with no gaps (1-2-2-3).

**Q5. How do you get the top 3 rows in each group?**
Sort descending and use `groupby(...).head(3)`, or compute `groupby(...)["col"].rank(method="first", ascending=False) <= 3`.

**Q6. `map` vs `apply` vs `applymap`/`DataFrame.map`?**
`Series.map` works element-wise (dict, function or Series), `Series.apply` also element-wise but accepts extra arguments, `DataFrame.apply` works on whole columns (`axis=0`) or rows (`axis=1`), and `DataFrame.map` (formerly `applymap`) works on every cell.

**Q7. `agg` vs `transform`?**
`agg` reduces each group to one value (output shrinks). `transform` returns a value for every original row (same shape), so it is used to add group statistics as features.

**Q8. Why is `apply(axis=1)` slow and what do you use instead?**
It calls a Python function once per row and builds a Series each time. Use vectorized column operations, `np.where`/`np.select`, or `zip` over columns.

**Q9. What is `shift` used for?**
Moving values down (positive) or up (negative) to create lag and lead features, and to compute changes against the previous row.

**Q10. What is the difference between `rolling` and `expanding`?**
`rolling(n)` uses a fixed-size window of the last `n` rows, `expanding()` uses all rows from the start up to the current row.

**Q11. How do you avoid data leakage with rolling features?**
Shift by one before rolling so the window only includes past values, and split train/test by time.

**Q12. Why must you sort before using `shift`, `diff` or `rolling`?**
They work on row position, not on dates. Unsorted data compares unrelated rows. Also use `groupby` so each group is processed separately.

---
# 11. Practice questions

Run this cell first to get the practice data.

In [116]:
emp = pd.DataFrame({
    "name":   ["Amit", "Bina", "Chirag", "Divya", "Esha", "Farhan", "Gita", "Harsh", "Isha", "Jay"],
    "dept":   ["ML", "Web", "ML", "Data", "Web", "ML", "Data", "Web", "ML", "Data"],
    "salary": [12, 18, 15, 9, 18, 16, 20, 11, 22, 13],                  # LPA, note the tie: Bina and Esha
    "rating": ["good", "excellent", "average", "good", "average", "excellent", "good", "poor", "excellent", "average"],
})

daily = pd.DataFrame({
    "date": pd.date_range("2026-04-01", periods=8),
    "revenue": [100, 120, 90, 150, 130, 160, 140, 170],
})
emp

,name,dept,salary,rating
0,Amit,ML,12,good
1,Bina,Web,18,excellent
2,Chirag,ML,15,average
3,Divya,Data,9,good
4,Esha,Web,18,average
5,Farhan,ML,16,excellent
6,Gita,Data,20,good
7,Harsh,Web,11,poor
8,Isha,ML,22,excellent
9,Jay,Data,13,average


**Q1.** Sort employees by `dept` A to Z, and inside each department by `salary` high to low. Reset the index.
**Q2.** Sort by `rating` in the order poor < average < good < excellent, ties broken by `salary` high to low.
**Q3.** Show the top 2 earners of every department (use `rank` with `method="first"`).
**Q4.** Add salary ranks (highest salary = 1) using `method="min"` and `method="dense"` side by side. What do Bina and Esha get, and what does the next person get in each?
**Q5.** Add `salary_pct`, the percentile rank of salary.
**Q6.** Add `salary_z_dept`: the z-score of salary inside each department (use `transform`).
**Q7.** Add `vs_dept_avg` = salary minus the average salary of the employee's department.
**Q8.** Create `label` = `"name (dept)"` with `apply(axis=1)`, then again with a vectorized version, and check that both are equal.
**Q9.** On `daily`: add `change` (difference from the previous day), `change_pct` and `roll3` (3-day rolling mean, use `min_periods=1`).
**Q10.** Add `running_total` and `running_max` of revenue.
**Q11.** Add `yesterday` (lag 1) and `beat_yesterday` (True if today is higher than yesterday).
**Q12.** Build model-ready data from `daily`: features `lag_1` and `roll3_past` (shift first, then roll with window 3), target `revenue`. Drop rows with NaN and split by time: last 2 rows are the test set.

### Solutions

In [117]:
# Q1
emp.sort_values(["dept", "salary"], ascending=[True, False], ignore_index=True)

,name,dept,salary,rating
0,Gita,Data,20,good
1,Jay,Data,13,average
2,Divya,Data,9,good
3,Isha,ML,22,excellent
4,Farhan,ML,16,excellent
5,Chirag,ML,15,average
6,Amit,ML,12,good
7,Bina,Web,18,excellent
8,Esha,Web,18,average
9,Harsh,Web,11,poor


In [118]:
# Q2
order = {"poor": 0, "average": 1, "good": 2, "excellent": 3}
(
    emp.sort_values(["rating", "salary"], ascending=[True, False],
                    key=lambda s: s.map(order) if s.name == "rating" else s)
       .reset_index(drop=True)
)

,name,dept,salary,rating
0,Harsh,Web,11,poor
1,Esha,Web,18,average
2,Chirag,ML,15,average
3,Jay,Data,13,average
4,Gita,Data,20,good
5,Amit,ML,12,good
6,Divya,Data,9,good
7,Isha,ML,22,excellent
8,Bina,Web,18,excellent
9,Farhan,ML,16,excellent


In [119]:
# Q3
r = emp.groupby("dept")["salary"].rank(method="first", ascending=False)
emp[r <= 2].sort_values(["dept", "salary"], ascending=[True, False])

,name,dept,salary,rating
6,Gita,Data,20,good
9,Jay,Data,13,average
8,Isha,ML,22,excellent
5,Farhan,ML,16,excellent
1,Bina,Web,18,excellent
4,Esha,Web,18,average


In [120]:
# Q4
q4 = emp.assign(
    rank_min=emp["salary"].rank(method="min", ascending=False).astype(int),
    rank_dense=emp["salary"].rank(method="dense", ascending=False).astype(int),
).sort_values("salary", ascending=False)
q4[["name", "salary", "rank_min", "rank_dense"]]
# Bina and Esha: both 3 (min) / both 3 (dense). The next person (Farhan, 16): 5 with "min", 4 with "dense".

,name,salary,rank_min,rank_dense
8,Isha,22,1,1
6,Gita,20,2,2
1,Bina,18,3,3
4,Esha,18,3,3
5,Farhan,16,5,4
2,Chirag,15,6,5
9,Jay,13,7,6
0,Amit,12,8,7
7,Harsh,11,9,8
3,Divya,9,10,9


In [121]:
# Q5
emp.assign(salary_pct=emp["salary"].rank(pct=True).round(2))[["name", "salary", "salary_pct"]].sort_values("salary")

,name,salary,salary_pct
3,Divya,9,0.10
7,Harsh,11,0.20
0,Amit,12,0.30
9,Jay,13,0.40
2,Chirag,15,0.50
5,Farhan,16,0.60
1,Bina,18,0.75
4,Esha,18,0.75
6,Gita,20,0.90
8,Isha,22,1.00


In [122]:
# Q6
e = emp.copy()
e["salary_z_dept"] = e.groupby("dept")["salary"].transform(lambda s: (s - s.mean()) / s.std()).round(2)
e[["name", "dept", "salary", "salary_z_dept"]].sort_values("dept")

,name,dept,salary,salary_z_dept
3,Divya,Data,9,-0.90
6,Gita,Data,20,1.08
9,Jay,Data,13,-0.18
0,Amit,ML,12,-1.01
2,Chirag,ML,15,-0.30
5,Farhan,ML,16,-0.06
8,Isha,ML,22,1.37
1,Bina,Web,18,0.58
4,Esha,Web,18,0.58
7,Harsh,Web,11,-1.15


In [123]:
# Q7
e["vs_dept_avg"] = (e["salary"] - e.groupby("dept")["salary"].transform("mean")).round(2)
e[["name", "dept", "salary", "vs_dept_avg"]]

,name,dept,salary,vs_dept_avg
0,Amit,ML,12,-4.25
1,Bina,Web,18,2.33
2,Chirag,ML,15,-1.25
3,Divya,Data,9,-5.00
4,Esha,Web,18,2.33
5,Farhan,ML,16,-0.25
6,Gita,Data,20,6.00
7,Harsh,Web,11,-4.67
8,Isha,ML,22,5.75
9,Jay,Data,13,-1.00


In [124]:
# Q8
slow = emp.apply(lambda r: f"{r['name']} ({r['dept']})", axis=1)
fast = emp["name"] + " (" + emp["dept"] + ")"
print(slow.equals(fast))
fast.head(3).tolist()

True


['Amit (ML)', 'Bina (Web)', 'Chirag (ML)']

In [125]:
# Q9
d = daily.copy()
d["change"] = d["revenue"].diff()
d["change_pct"] = (d["revenue"].pct_change() * 100).round(1)
d["roll3"] = d["revenue"].rolling(3, min_periods=1).mean().round(1)
d

,date,revenue,change,change_pct,roll3
0,2026-04-01,100,NaN,NaN,100.0
1,2026-04-02,120,20.0,20.0,110.0
2,2026-04-03,90,-30.0,-25.0,103.3
3,2026-04-04,150,60.0,66.7,120.0
4,2026-04-05,130,-20.0,-13.3,123.3
5,2026-04-06,160,30.0,23.1,146.7
6,2026-04-07,140,-20.0,-12.5,143.3
7,2026-04-08,170,30.0,21.4,156.7


In [126]:
# Q10
d["running_total"] = d["revenue"].cumsum()
d["running_max"] = d["revenue"].cummax()
d[["date", "revenue", "running_total", "running_max"]]

,date,revenue,running_total,running_max
0,2026-04-01,100,100,100
1,2026-04-02,120,220,120
2,2026-04-03,90,310,120
3,2026-04-04,150,460,150
4,2026-04-05,130,590,150
5,2026-04-06,160,750,160
6,2026-04-07,140,890,160
7,2026-04-08,170,1060,170


In [127]:
# Q11
d["yesterday"] = d["revenue"].shift(1)
d["beat_yesterday"] = d["revenue"] > d["yesterday"]       # first row: NaN comparison gives False
d[["date", "revenue", "yesterday", "beat_yesterday"]]

,date,revenue,yesterday,beat_yesterday
0,2026-04-01,100,NaN,False
1,2026-04-02,120,100.0,True
2,2026-04-03,90,120.0,False
3,2026-04-04,150,90.0,True
4,2026-04-05,130,150.0,False
5,2026-04-06,160,130.0,True
6,2026-04-07,140,160.0,False
7,2026-04-08,170,140.0,True


In [128]:
# Q12
m = daily.copy()
m["lag_1"] = m["revenue"].shift(1)
m["roll3_past"] = m["revenue"].shift(1).rolling(3).mean()
m = m.dropna().reset_index(drop=True)

train, test = m.iloc[:-2], m.iloc[-2:]
print("train rows:", len(train), "| test rows:", len(test))
m

train rows: 3 | test rows: 2


,date,revenue,lag_1,roll3_past
0,2026-04-04,150,90.0,103.333333
1,2026-04-05,130,150.0,120.000000
2,2026-04-06,160,130.0,123.333333
3,2026-04-07,140,160.0,146.666667
4,2026-04-08,170,140.0,143.333333


---
# 12. Cheat sheet

```python
# SORT
df.sort_values("a")                                  df.sort_values("a", ascending=False)
df.sort_values(["a", "b"], ascending=[True, False])  df.sort_values("a", kind="stable", ignore_index=True)
df.sort_values("a", na_position="first")
df.sort_values("t", key=lambda s: s.str.lower())     df.sort_values("lvl", key=lambda s: s.map(order))
df.sort_index()    df.nlargest(3, "a")    df.nsmallest(3, "a", keep="all")
df.sort_values("a", ascending=False).groupby("g").head(2)       # top 2 per group

# RANK
s.rank(ascending=False, method="min")     # average | min | max | first | dense
s.rank(pct=True)                          # percentile
df.groupby("g")["a"].rank(method="first", ascending=False)

# MAP / APPLY / TRANSFORM
s.map({"a": 1}) / s.map(func)             s.apply(func, arg=...)
df.apply(func)                            # per column
df.apply(func, axis=1)                    # per row (slow)
df.map(func)                              # per cell (was applymap)
df.groupby("g")["a"].agg("mean")          # shrinks
df.groupby("g")["a"].transform("mean")    # same shape  -> features
df.pipe(func, arg)

# SEQUENTIAL (sort by group + time first!)
g = df.groupby("store")["x"]
g.cumsum()  g.cummax()  g.diff()  g.pct_change()  g.shift(1)  g.shift(-1)
g.transform(lambda s: s.rolling(3, min_periods=1).mean())
g.transform(lambda s: s.shift(1).rolling(3).mean())      # past-only (no leakage)
s.rolling("3D").mean()      s.expanding().mean()      s.ewm(span=3).mean()

# SPEED: vectorized > np.where/np.select > map(dict) > apply(col) > zip > apply(axis=1) > iterrows
# SPEED — typical preference
# Prefer vectorized operations first.
# Then np.where/np.select or map(dict) when appropriate.
# For custom Python logic, prefer column-wise apply or itertuples/zip
# over slow row-wise methods such as apply(axis=1) and iterrows.
```

**Next chapter:** `10_groupby_aggregation`: split-apply-combine, `agg` with several functions, named aggregation, multiple keys, `filter`, `size` vs `count`, and group statistics for features.